In [ ]:
VER = 6  # 脚本版本

import pandas as pd, numpy as np  # 导入数据处理和数值计算库
from tqdm.notebook import tqdm  # Jupyter 环境下的进度条
tqdm.pandas()  # 为 pandas 操作启用进度条支持

import os, sys, pickle, glob, gc  # 导入常用标准库
from collections import Counter  # 导入计数工具
import cudf, itertools  # 导入 RAPIDS 和迭代工具
print('We will use RAPIDS version',cudf.__version__)  # 打印 RAPIDS 版本

pd.set_option('display.max_rows', 500)  # 设置最大显示行数
pd.set_option('display.max_columns', 500)  # 设置最大显示列数

from pandarallel import pandarallel  # pandas 并行处理工具

pandarallel.initialize(nb_workers=4, progress_bar=True, use_memory_fs=False)  # 初始化并行环境


# Generating Splits

In [ ]:
type_labels = {'clicks':0, 'carts':1, 'orders':2}  # 将行为类型映射为整数标签


In [ ]:
def load_test():  # 读取并整理测试集分片
    dfs = []  # 保存每个分片的数据
    for e, chunk_file in enumerate(glob.glob('../raw_data/test_parquet/*')):  # 遍历测试集 parquet 文件
        chunk = pd.read_parquet(chunk_file)  # 读取单个分片
        chunk.ts = (chunk.ts/1000).astype('int32')  # 将毫秒时间戳转换为秒并压缩类型
        chunk['type'] = chunk['type'].map(type_labels).astype('int8')  # 将行为类型编码为整数
        dfs.append(chunk)  # 收集当前分片
    return pd.concat(dfs).reset_index(drop=True) #.astype({"ts": "datetime64[ms]"})  # 合并所有分片并重置索引

test_df = load_test()  # 读取测试集
test_df.to_parquet("./splitted_raw_data/test.parquet")  # 保存处理后的测试集


In [ ]:
def load_train():  # 读取并整理训练集分片
    dfs = []  # 保存每个分片的数据
    for e, chunk_file in enumerate(glob.glob('../raw_data/train_parquet/*')):  # 遍历训练集 parquet 文件
        chunk = pd.read_parquet(chunk_file)  # 读取单个分片
        chunk.ts = (chunk.ts/1000).astype('int32')  # 将毫秒时间戳转换为秒并压缩类型
        chunk['type'] = chunk['type'].map(type_labels).astype('int8')  # 将行为类型编码为整数
        dfs.append(chunk)  # 收集当前分片
    return pd.concat(dfs).reset_index(drop=True) #.astype({"ts": "datetime64[ms]"})  # 合并所有分片并重置索引

all_train_df = load_train()  # 读取全部训练集
all_train_df.to_parquet("./splitted_raw_data/all_train.parquet")  # 保存处理后的全部训练集


In [ ]:
train_df = pd.read_parquet("../raw_data/radek/train.parquet")  # 读取现成的训练集
train_df.ts = train_df.ts.astype('int32')  # 将时间戳转换为 int32
train_df.to_parquet("./splitted_raw_data/train.parquet")  # 保存训练集

val_df = pd.read_parquet("../raw_data/radek/test.parquet")  # 读取现成的验证集来源数据
val_df.ts = val_df.ts.astype('int32')  # 将时间戳转换为 int32

np.random.seed(1337)  # 固定随机种子，保证划分可复现
val_sess = np.random.choice(val_df.session.unique(), val_df.session.nunique()//2)  # 随机抽取一半会话作为验证会话

val_df.to_parquet("./splitted_raw_data/val.parquet")  # 保存验证集


In [ ]:
val_sess.dump("./splitted_raw_data/val_sessions_for_train.npy")  # 保存验证会话列表


In [ ]:
val_labels_df = pd.read_parquet("../raw_data/radek/test_labels.parquet")  # 读取验证标签
val_labels_df.to_parquet("./splitted_raw_data/val_labels.parquet")  # 保存验证标签
